# Notebook 4: Fine-Tuning

**Goal:** Fine-tune the sentence-transformer model on Food.com interaction data and measure whether domain adaptation improves retrieval quality.

**Research question:** Does domain-specific fine-tuning on food interaction data improve recommendation quality, even with proxy labels (ratings vs. actual orders)?

**Requires:** GPU runtime. Go to *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import torch
assert torch.cuda.is_available(), "⚠️  This notebook requires GPU. Go to Runtime > Change runtime type > GPU"
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# DATA PATHS - auto-detected for Colab and Kaggle
# ============================================================
import os

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'

if ENV == 'kaggle':
    PROCESSED_DIR = '/kaggle/working/processed'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROCESSED_DIR = '/content/drive/MyDrive/food-recsys/processed'

print(f"Reading data from: {PROCESSED_DIR}")

In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, sys

REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'
BASE = '/kaggle/working' if ENV == 'kaggle' else '/content'
REPO_DIR = f'{BASE}/{REPO}'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull -q')

os.system('pip install -q sentence-transformers faiss-cpu pandas pyarrow matplotlib seaborn umap-learn plotly scikit-learn tqdm')
sys.path.insert(0, f'{REPO_DIR}/src')
print(f"Environment: {ENV} | Repo: {REPO_DIR}")
print("Setup complete")

In [ ]:
recipes = pd.read_parquet(f'{PROCESSED_DIR}/recipes.parquet')
train = pd.read_parquet(f'{PROCESSED_DIR}/interactions_train.parquet')
test = pd.read_parquet(f'{PROCESSED_DIR}/interactions_test.parquet')

recipe_id_to_idx = {rid: i for i, rid in enumerate(recipes['id'])}
idx_to_recipe_id = {i: rid for i, rid in enumerate(recipes['id'])}

# Use best text config from NB3
id_to_text = {
    row['id']: dish_to_rich_text(row.to_dict(), tags=row.get('tag_list', []),
        include_recipe=False, include_macro_tokens=False, include_ratios=True, include_ingredients=True)
    for _, row in recipes.iterrows()
}

with open(f'{PROCESSED_DIR}/results.json') as f:
    all_results = json.load(f)

print(f"Loaded {len(recipes):,} recipes")

## 1. Training Pair Construction

We use **MultipleNegativesRankingLoss** with (anchor, positive, hard_negative) triplets.

**Triplet strategy:**
- **Anchor:** Text of a recipe rated ≥4 by user U
- **Positive:** Text of ANOTHER recipe rated ≥4 by the SAME user U (co-preference)
- **Hard Negative:** A recipe that FAISS retrieves as similar to the anchor, but user U did NOT rate

**Important caveat:** "not rated" ≠ "disliked." The user may not have seen the recipe. Hard negative mining is standard practice in retrieval fine-tuning, but the noise is real — we acknowledge this limitation.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample

# Load pretrained model
base_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# Build pretrained embeddings for hard negative mining
print("Building pretrained embeddings for hard negative mining...")
texts_list = [id_to_text[rid] for rid in recipes['id'] if rid in id_to_text]
ids_list = [rid for rid in recipes['id'] if rid in id_to_text]

pretrained_embs = base_model.encode(texts_list, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
mining_index = faiss.IndexFlatIP(pretrained_embs.shape[1])
mining_index.add(pretrained_embs.astype(np.float32))
mining_id_to_idx = {rid: i for i, rid in enumerate(ids_list)}
mining_idx_to_id = {i: rid for i, rid in enumerate(ids_list)}

print(f"Mining index: {mining_index.ntotal:,} vectors")

In [ ]:
user_train_positives = (
    train[train['rating'] >= 4]
    .groupby('user_id')['recipe_id']
    .apply(list)
    .to_dict()
)

train_examples = []
MAX_TRIPLETS = 50000  # limit for Colab

for user_id, pos_list in user_train_positives.items():
    # Only users with enough positives
    pos_set = {r for r in pos_list if r in id_to_text}
    if len(pos_set) < 10:
        continue

    pos_list_clean = list(pos_set)
    for i in range(min(3, len(pos_list_clean) - 1)):
        anchor_id = pos_list_clean[i]
        positive_id = pos_list_clean[i + 1]

        # Hard negative: top FAISS result that user hasn't rated
        anchor_idx = mining_id_to_idx.get(anchor_id)
        if anchor_idx is None:
            continue

        _, faiss_indices = mining_index.search(pretrained_embs[anchor_idx:anchor_idx+1], 20)
        hard_neg_id = None
        for idx in faiss_indices[0]:
            candidate_id = mining_idx_to_id.get(idx)
            if candidate_id and candidate_id not in pos_set and candidate_id != anchor_id:
                hard_neg_id = candidate_id
                break

        if hard_neg_id is None:
            continue

        train_examples.append(InputExample(
            texts=[id_to_text[anchor_id], id_to_text[positive_id], id_to_text[hard_neg_id]]
        ))

    if len(train_examples) >= MAX_TRIPLETS:
        break

print(f"Training triplets: {len(train_examples):,}")
print(f"\nExample triplet:")
ex = train_examples[0]
print(f"  Anchor:   {ex.texts[0][:80]}")
print(f"  Positive: {ex.texts[1][:80]}")
print(f"  Hard neg: {ex.texts[2][:80]}")

## 2. Fine-Tuning

In [ ]:
from sentence_transformers import losses
from torch.utils.data import DataLoader

EPOCHS = 3
BATCH_SIZE = 32
WARMUP_STEPS = 100

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.MultipleNegativesRankingLoss(base_model)

print(f"Fine-tuning: {len(train_examples):,} triplets | {EPOCHS} epochs | batch={BATCH_SIZE}")
base_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path='models/food-recsys-finetuned',
    show_progress_bar=True,
)
print("✓ Fine-tuning complete")

## 3. Evaluate: Pretrained vs Fine-tuned

In [ ]:
def evaluate_retrieval(index, embeddings, ks=None):
    if ks is None: ks = [5, 10, 20]
    user_positives = (test[test['rating'] >= 4].groupby('user_id')['recipe_id'].apply(set).to_dict())
    results = []
    for user_id, pos_recipes in user_positives.items():
        pos_recipes = {r for r in pos_recipes if r in recipe_id_to_idx}
        if len(pos_recipes) < 5: continue
        query_recipe = list(pos_recipes)[0]
        relevant = pos_recipes - {query_recipe}
        query_idx = recipe_id_to_idx[query_recipe]
        scores, indices = index.search(embeddings[query_idx:query_idx+1], max(ks) + 1)
        recommended = [idx_to_recipe_id[i] for i in indices[0] if i >= 0 and idx_to_recipe_id.get(i) != query_recipe]
        results.append(evaluate_all(recommended, relevant, ks=ks))
    return pd.DataFrame(results).mean().to_dict() if results else {}

# Fine-tuned model embeddings
finetuned_model = SentenceTransformer('models/food-recsys-finetuned')
finetuned_embs = finetuned_model.encode(texts_list, batch_size=64, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

finetuned_index = faiss.IndexFlatIP(finetuned_embs.shape[1])
finetuned_index.add(finetuned_embs)

metrics_finetuned = evaluate_retrieval(finetuned_index, finetuned_embs)
all_results['finetuned'] = metrics_finetuned

# Best pretrained result from NB3
metrics_best_pretrained = all_results.get('full_improved', {})

comparison = pd.DataFrame({
    'Best Pretrained': metrics_best_pretrained,
    'Fine-tuned': metrics_finetuned,
}).T
print("=== Pretrained vs Fine-tuned ===")
print(comparison[['P@5', 'P@10', 'NDCG@10', 'MRR']].round(4))

delta_p10 = metrics_finetuned.get('P@10', 0) - metrics_best_pretrained.get('P@10', 0)
print(f"\nΔ P@10: {delta_p10:+.4f}")

## 4. UMAP: Pretrained vs Fine-tuned

In [ ]:
SAMPLE = 2000
sample_idx = np.random.choice(len(pretrained_embs), SAMPLE, replace=False)

id_to_tags = dict(zip(recipes['id'], recipes['tag_list']))
sample_ids = [ids_list[i] for i in sample_idx]

labels = []
for rid in sample_ids:
    tags = id_to_tags.get(rid, [])
    label = next((t for t in tags if t in ['breakfast','lunch','dinner','snacks']), 'other')
    labels.append(label)

fig, axes = __import__('matplotlib.pyplot', fromlist=['subplots']).subplots(1, 2, figsize=(16, 6))

import matplotlib.pyplot as plt
import umap

for ax, embs, title in [
    (axes[0], pretrained_embs[sample_idx], 'Pretrained Embeddings'),
    (axes[1], finetuned_embs[sample_idx], 'Fine-tuned Embeddings'),
]:
    coords = umap.UMAP(n_components=2, random_state=42, metric='cosine').fit_transform(embs)
    unique = list(set(labels))
    cmap = plt.cm.get_cmap('tab10', len(unique))
    for i, lab in enumerate(unique):
        mask = [j for j, l in enumerate(labels) if l == lab]
        ax.scatter(coords[mask, 0], coords[mask, 1], s=6, alpha=0.5, color=cmap(i), label=lab)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Fine-tuned + all improvements combined
# (fine-tuned model already uses best text config)
all_results['finetuned_full'] = metrics_finetuned

with open(f'{PROCESSED_DIR}/results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
np.save(f'{PROCESSED_DIR}/embeddings_finetuned.npy', finetuned_embs)
print("✓ Saved")